# Earlier findings of peterson-2021-01, checked on DANDI:000055 version 0.261123.1703

Paper. Peterson SM, Singh SH, Wang NXR, Rao RPN, Brunton BW (2021). Behavioral and neural
variability of naturalistic arm movements. eNeuro 8(3). [doi:10.1523/eneuro.0007-21.2021](https://doi.org/10.1523/eneuro.0007-21.2021)

Dataset. [DANDI:000055](https://dandiarchive.org/dandiset/000055/0.261123.1703) (AJILE12), version
0.261123.1703, streamed with the entry's helper `ajile.py`. The earlier attempt, peterson-2021-01,
used version 0.220127.0436.

The repeat attempt peterson-2021-02 reports most of the earlier findings again. This notebook checks
the four it does not report (1, 6, 8 and 11), one section each, opening with the earlier statement.
A last section reruns the precentral power of the repeat attempt's helper with the ECoG read from its
`starting_time`, for the two outcomes on which the attempts differ.

In [ ]:
import os, time, zlib
os.environ.setdefault("OMP_NUM_THREADS", "1")
import numpy as np
import pandas as pd
from scipy import stats
import ajile

pd.set_option("display.width", 200)
T0 = time.time()
assets = ajile.list_assets()
metas = ajile.load_all_meta(assets)
subjects = sorted({int(m["subject_id"]) for m in metas})
print("dandiset", ajile.DANDISET, "version", ajile.VERSION, ";", len(assets), "files,", len(subjects), "subjects,",
      sum(len(m["reaches"]) for m in metas), "events")

## Earlier finding 1

> In three files (sub-01 ses-7, sub-04 ses-3, sub-06 ses-3) the ECoG holds 15.5 to 15.9 h and its
> starting_time is 0, while the reaches table and the pose run on a clock from midnight. The ECoG
> starts 29241 to 30682 s after midnight, the time missing from 24 h; ReachEvents and the session
> start time agree, and the movement response appears only with this shift (S06 day 3, precentral
> HFB +1.98 dB with it, -0.07 dB without). Read with starting_time as given, the event times point at
> the wrong ECoG or past its end.

For every file, the ECoG `starting_time`, its length, and whether it ends at 24 h; whether the
ReachEvents times equal the reaches table's start_time; and, for S06 day 3, the event-locked
precentral band power (baseline -1.5 to -1 s, 0 to 0.5 s, the helper's method and the electrodes
good in every file of the subject) with the ECoG read from its `starting_time` and from 0 s.

In [ ]:
rows = []
for m in metas:
    n = m["ecog_shape"][0] / m["ecog_rate"]
    rows.append(dict(subject=int(m["subject_id"]), day=int(m["session_id"]), ecog_start_s=m["ecog_start"], ecog_hours=round(n / 3600, 3),
                     ecog_end_s=round(m["ecog_start"] + n, 1), pose_start_s=m["pose_start"],
                     reachevents_vs_start_time_max_s=round(float(np.abs(m["event_times"] - m["reaches"].start_time.values).max()), 3)))
clock = pd.DataFrame(rows)
print(clock[clock.ecog_start_s != 0].to_string(index=False))
print("files with starting_time not 0:", int((clock.ecog_start_s != 0).sum()), "; files whose ECoG ends at 86400 s:",
      int((clock.ecog_end_s.round() == 86400).sum()), "of", len(clock))
print("files where ReachEvents and start_time differ by more than 1 s:",
      clock[clock.reachevents_vs_start_time_max_s > 1][["subject", "day", "reachevents_vs_start_time_max_s"]].values.tolist())

In [ ]:
files, _ = ajile.figshare_files()
W = {}
for s in subjects:
    pm, _ = ajile.load_projection(s, files)
    ms = [m for m in metas if int(m["subject_id"]) == s]
    a, _ = ajile.match_projection(pm, ms[0]["electrodes"].x.notna().values)
    roi_idx = [list(pm.columns).index(r) for r in ajile.ROIS]
    G = np.array([m["electrodes"].good.values.astype(bool) for m in ms])
    W[s] = dict(proj=a[:, roi_idx], G=G)

m6 = [m for m in metas if int(m["subject_id"]) == 6 and int(m["session_id"]) == 3][0]
good6 = W[6]["G"].all(0)
w6 = ajile.session_weights(W[6]["proj"], good6)
for label, t in (("ECoG read from its starting_time", m6["event_times"] - m6["ecog_start"]),
                 ("ECoG read from 0 s", m6["event_times"])):
    r = ajile.session_spectral(m6["asset_id"], t, good6, w6)
    pre = r["region"][:, 1]
    st = ajile.band_stats(pre - pre[..., ajile.BASE].mean(axis=-1, keepdims=True))
    ok = r["inside"] & np.isfinite(st["hfb_mean"])
    print(f"S06 day 3, {label}: {int(ok.sum())} of {len(t)} events with ECoG; precentral HFB median "
          f"{np.median(st['hfb_mean'][ok]):+.2f} dB, LFB {np.median(st['lfb_mean'][ok]):+.2f} dB")

Result: fixed. The three files now give the ECoG `starting_time` as 30682.414 s (sub-01 day 7),
30048.322 s (sub-04 day 3) and 29240.566 s (sub-06 day 3), so their ECoG ends at 86400 s, on the clock
of the reaches table, ReachEvents and the pose; the other 52 files start at 0 s. Read from its
`starting_time`, the S06 day 3 ECoG shows the movement response; read from 0 s it does not (the
numbers are printed above).

## Earlier finding 6

> session_start_time is 19:00 at UTC-5 for files that start at local midnight (midnight UTC written
> in another zone), and 03:07 to 03:31 at UTC-5 for the three files whose ECoG starts later.

In [ ]:
sst = pd.DataFrame([dict(subject=int(m["subject_id"]), day=int(m["session_id"]), session_start_time=m["session_start"]) for m in metas])
print("times of day and zones of session_start_time:", sst.session_start_time.str[10:].value_counts().to_dict())
print(sst[(sst.subject.isin([1, 4, 6]))].to_string(index=False))

Result: fixed. All 55 files give session_start_time at 00:00:00+00:00 of their day, including the
three files whose ECoG starts later, which now carry that delay in the ECoG `starting_time`
(finding 1).

## Earlier finding 8

> The subjects' sex and age, which the paper reports, are not in the files or the archive's metadata.

In [ ]:
subj = pd.DataFrame([dict(subject=int(m["subject_id"]), sex=m["sex"], age=m["age"]) for m in metas]).drop_duplicates()
print(subj.to_string(index=False))
print("subjects with more than one sex or age across files:", int((subj.groupby("subject").size() > 1).sum()))
ages = subj.age.str.extract(r"P(\d+)Y")[0].astype(float)
print("male:", int((subj.sex == "M").sum()), "female:", int((subj.sex == "F").sum()),
      f"; age mean {ages.mean():.2f} SD {ages.std():.2f} (paper: 8 male, 4 female, 29.4 +- 7.9 years)")

Result: fixed. Every file gives the subject's sex and age (ISO 8601 durations); 8 subjects are male
and 4 female, with a mean age of 29.42 years (SD 7.94), the paper's values.

## Earlier finding 11

> The electrode group descriptions and the data descriptor's Table 2 disagree for two subjects. S11
> has groups LATD, LMTD and LTPD described as depth (18 electrodes), where the descriptor gives S11
> 106 surface and no depth electrodes; S10 has 78 electrodes in groups not described as depth and 48
> in depth groups, where the descriptor gives 86 and 40.

The groups of S10 and S11 with their descriptions and electrode counts, in each of their files. The
descriptor's counts are those the earlier statement quotes (doi 10.1038/s41597-022-01280-y, Table 2).

In [ ]:
for s in (10, 11):
    for m in [m for m in metas if int(m["subject_id"]) == s]:
        el = m["electrodes"]
        desc = el.group_name.map(m["groups"]).fillna("")
        dep = desc.str.contains("depth", case=False)
        if int(m["session_id"]) == 3:
            print(f"S{s} groups:", el.assign(description=desc).groupby(["group_name", "description"]).size().to_dict())
        print(f"S{s} day {m['session_id']}: {int((~dep).sum())} electrodes in groups not described as depth, {int(dep.sum())} in depth groups")

Result: still present. In every file S10 has 78 electrodes in groups not described as depth and 48 in
groups described as depth (AHD, AID, MHD, MID, PID), and S11 has 88 and 18 (LATD, LMTD and LTPD, each
described as depth), against the descriptor's 86 and 40, and 106 and 0.

## The ECoG starting_time and two outcomes of peterson-2021-02

The repeat attempt's helper places each event at sample `event time x 500` of the ECoG
(`ajile.session_spectral`, "series starts at 0 s"), which ignores the `starting_time` of the three
files of finding 1. Here every file's precentral power is computed with the same helper and the same
rules for keeping events (`reproduce_peterson_2021.ipynb`, cell 12), with the event times less the
`starting_time`, under four sets of good electrodes: good in every file of the subject (the repeat
attempt's main set), each file's own flags (the set its outcome of fig5-recording-day-effects rests
on), good in at least one file, and the first file's flags (the earlier attempt's set, as the paper's
code takes them). For each set it gives the parts of fig4-precentral-subject-variability and the
count of fig5-recording-day-effects, as the claims file's criteria define them.

In [ ]:
t = time.time()
jobs, keys = [], []
for s in subjects:
    proj, G = W[s]["proj"], W[s]["G"]
    sets = {"good in every file": G.all(0), "good in at least one file": G.any(0), "first file's flags": G[0]}
    wv0 = {k: ajile.session_weights(proj, g) for k, g in sets.items()}
    for i, m in enumerate([m for m in metas if int(m["subject_id"]) == s]):
        wv = dict(wv0)
        wv["each file's flags"] = ajile.session_weights(proj, G[i])
        jobs.append((m["asset_id"], m["event_times"] - m["ecog_start"], G.all(0), wv["good in every file"], wv))
        keys.append((s, int(m["session_id"])))
spec = ajile.run_spectral(jobs, workers=8)
print(f"spectral pass {time.time() - t:.0f} s")

In [ ]:
E = pd.concat([pd.DataFrame(dict(subject=s, day=d, inside=r["inside"], allzero=r["allzero"], seg_db=r["seg_db"], hf_db=r["hf_db"],
                                 mism=np.abs(m["event_times"] - m["reaches"].start_time.values) > 1))
               for (s, d), r, m in zip(keys, spec, [m for s in subjects for m in metas if int(m["subject_id"]) == s])], ignore_index=True)
ok = E.inside & E.hf_db.notna()
thr = E[ok].groupby("subject").hf_db.agg(lambda x: x.mean() + 3 * x.std())
E["keep"] = E.inside & E.seg_db.notna() & ~E.allzero & ~(E.seg_db < 0) & ~(E.hf_db > E.subject.map(thr)) & ~E.mism
print("events kept:", int(E.keep.sum()), "of", len(E), "; per subject:", E[E.keep].groupby("subject").size().to_dict())
print("kept in the three files of finding 1:", {f"S{s:02d} day {d}": int(E[(E.subject == s) & (E.day == d)].keep.sum()) for s, d in ((1, 7), (4, 3), (6, 3))})
F = ajile.FREQS
paper_sig = {"lfb": {3, 5, 6, 7, 8, 11}, "hfb": {3, 4, 7, 8, 10}}
kb, sv = E.keep.values, E.subject.values
for name in ("good in every file", "each file's flags", "good in at least one file", "first file's flags"):
    R = np.concatenate([r["region_var"][name][:, 1] for r in spec])
    RB = R - R[..., ajile.BASE].mean(axis=-1, keepdims=True)
    SM = np.stack([np.nanmedian(RB[kb & (sv == s)], axis=0) for s in subjects])
    P = SM - SM[..., ajile.BASE].mean(axis=-1, keepdims=True)
    sp = P[..., ajile.WIN].mean(axis=-1)
    lf, hf = sp[:, ajile.LFB].mean(1), sp[:, ajile.HFB].mean(1)
    part1 = [f"S{s:02d}" for s, a, b in zip(subjects, lf, hf) if not (a < 0 and b > 0)]
    i116, i121 = list(F).index(116), list(F).index(121)
    part2 = all(sp[subjects.index(s), i116] > 0 and sp[subjects.index(s), i121] > 0 for s in [3, 6, 7, 8, 11])
    part3 = {}
    for s in [9, 12]:
        x = sp[subjects.index(s)]; hi = F >= 56; fmax = F[hi][np.argmax(x[hi])]
        part3[f"S{s:02d}"] = (int(fmax), bool(61 <= fmax <= 76 and x[F >= 101].mean() < x[hi].max() / 2))
    width = {f"S{s:02d}": int((sp[i][F <= 46] < sp[i][F <= 46].min() / 2).sum()) for i, s in enumerate(subjects)}
    print(f"=== {name}")
    print(f"fig4 part 1, subjects without the pattern: {part1} (S09 76-100 Hz {hf[subjects.index(9)]:.3f} dB); part 2 {part2}; part 3 {part3}")
    print("fig4 part 4, bins below half the minimum:", width)
    bs = ajile.band_stats(RB)
    for st in ("mean", "median"):
        agree, dis = 0, []
        for s in subjects:
            for b in ("lfb", "hfb"):
                g = E[kb & (sv == s)].assign(v=bs[f"{b}_{st}"][kb & (sv == s)])
                p = stats.kruskal(*[x.v.values for _, x in g.groupby("day")]).pvalue
                row = (f"S{s:02d}", b, float(f"{p:.3g}"))
                if (p < 0.05) == (s in paper_sig[b]):
                    agree += 1
                else:
                    dis.append(row)
                if s in (1, 6) and st == "mean":
                    print("  ", row, "paper significant:", s in paper_sig[b])
        print(f"fig5-recording-day-effects, band {st}: {agree} of 24 pairs agree; disagreeing {dis}")
print(f"notebook run time {(time.time() - T0) / 60:.1f} min")

With the ECoG read from its `starting_time`, the day effects of S01 (LFB and HFB) and S06 (HFB),
which the repeat attempt found where the paper has none, are no longer significant (the p values of
S01 and S06 are printed above). The counts of fig5-recording-day-effects and the parts of
fig4-precentral-subject-variability under each set of electrodes are printed above; the
reconciliation in `attempts/peterson-2021-02.yaml` quotes them.